In [1]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
def map_query_ids(csv_micro_path, csv_query_path):
    df_gallery = pd.read_csv(csv_micro_path)  # gldv2_micro.csv
    df_queries_raw = pd.read_csv(csv_query_path)  # val.csv

    df_gallery.columns = df_gallery.columns.str.strip()
    df_queries_raw.columns = df_queries_raw.columns.str.strip()

    df_queries_filtered = df_queries_raw[df_queries_raw['landmark_id'].between(0, 99)]
    df_queries = df_queries_filtered.merge(df_gallery, on='filename', suffixes=('_bad', ''))

    unique_true_landmarks = df_queries['landmark_id'].unique()
    df_gallery = df_gallery[df_gallery['landmark_id'].isin(unique_true_landmarks)]
    
    print(f"Galerie geladen: {len(df_gallery)} Einträge.")
    print(f"Queries erfolgreich auf echte IDs umgemappt: {len(df_queries)} Einträge.")

    return df_gallery, df_queries

In [3]:
df_gallery, df_queries = map_query_ids("gldv2_micro/gldv2_micro.csv", "gldv2_micro/val.csv")

Galerie geladen: 847 Einträge.
Queries erfolgreich auf echte IDs umgemappt: 100 Einträge.


In [4]:
sift = cv2.SIFT_create(nfeatures=2000, edgeThreshold=10, contrastThreshold=0.04)
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))

In [5]:
def build_gallery(df_gallery, dataset_dir):
    print("\nIndiziere Galerie-Features...")
    gallery_features = []

    for idx, row in df_gallery.iterrows():
        img_file = row["filename"]
        true_l_id = row["landmark_id"]
        
        img_path = os.path.join(dataset_dir, str(img_file))
        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kp, des = sift.detectAndCompute(gray, None)

        if des is not None and len(des) > 10:
            des /= des.sum(axis=1, keepdims=True) + 1e-7
            des = np.sqrt(des)

            gallery_features.append(
                {"filename": img_file, "landmark_id": true_l_id, "kp": kp, "des": des}
            )

    print(f"Datenbank einsatzbereit! {len(gallery_features)} Bilder indiziert.")
    return gallery_features

In [6]:
gallery_features = build_gallery(df_gallery, "gldv2_micro/images")


Indiziere Galerie-Features...
Datenbank einsatzbereit! 847 Bilder indiziert.


In [7]:
def run_landmark_system(gallery_features, df_queries, dataset_dir, num_queries_to_test=100, localization=False):
    total_processed_queries = 0
    top1_correct_count = 0
    top5_correct_count = 0

    test_queries = df_queries.head(num_queries_to_test)

    for q_idx, q_row in test_queries.iterrows():
        q_file = q_row["filename"]
        true_landmark = q_row["landmark_id"]

        img_q_path = os.path.join(dataset_dir, str(q_file))
        q_img = cv2.imread(img_q_path)
        if q_img is None:
            print(f"Überspringe Query '{q_file}' (Bilddatei nicht gefunden).")
            continue

        q_gray = cv2.cvtColor(q_img, cv2.COLOR_BGR2GRAY)
        kp_q, des_q = sift.detectAndCompute(q_gray, None)

        if des_q is None:
            continue

        # RootSIFT für Query
        des_q /= des_q.sum(axis=1, keepdims=True) + 1e-7
        des_q = np.sqrt(des_q)

        print(f"\n==================================================")
        print(f"Starte Suche für Query-Bild: {q_file} (Echte Landmark ID: {true_landmark})")

        retrieval_results = []

        for g_data in gallery_features:
            # Gleiches Bild in der Galerie ausschließen
            if g_data["filename"] == q_file:
                continue

            matches = flann.knnMatch(des_q, g_data["des"], k=2)
            good_matches = [m for m, n in matches if m.distance < 0.70 * n.distance]

            retrieval_results.append(
                {
                    "filename": g_data["filename"],
                    "landmark_id": g_data["landmark_id"],
                    "score": len(good_matches),
                    "kp_g": g_data["kp"],
                    "matches": good_matches,
                }
            )

        retrieval_results = sorted(retrieval_results, key=lambda x: x["score"], reverse=True)

        if len(retrieval_results) == 0:
            continue

        total_processed_queries += 1

        # 1. Check Top-1 (Rang 1)
        top1_match = retrieval_results[0]
        if int(top1_match["landmark_id"]) == int(true_landmark):
            top1_correct_count += 1

        # 2. Check Top-5 (Ist die richtige ID unter den ersten 5?)
        top5_matches = retrieval_results[:5]
        top5_landmark_ids = [int(res["landmark_id"]) for res in top5_matches]

        if int(true_landmark) in top5_landmark_ids:
            top5_correct_count += 1

        print("\n--- Top 5 Retrieval Ergebnisse ---")
        for rank, res in enumerate(retrieval_results[:5]):
            is_correct = int(res["landmark_id"]) == int(true_landmark)
            status = "✅ KORREKT (Treffer)" if is_correct else "❌ ANDERE SEHENSWÜRDIGKEIT"
            print(f" Rang {rank+1}: Datei {res['filename']} | Landmark {res['landmark_id']} | Matches: {res['score']} | {status}")

        # --- LOKALISIERUNG (Top 1 Fund) ---
        if localization:
          top1_match = retrieval_results[0]
          if top1_match["score"] >= 4:
              pts_q = np.float32([kp_q[m.queryIdx].pt for m in top1_match["matches"]]).reshape(-1, 1, 2)
              pts_g = np.float32([top1_match["kp_g"][m.trainIdx].pt for m in top1_match["matches"]]).reshape(-1, 1, 2)

              # H_est, _ = cv2.findHomography(pts_q, pts_g, cv2.RANSAC, 5.0)
              H_est, _ = cv2.findHomography(pts_q, pts_g, cv2.USAC_MAGSAC, 3.0)

              if H_est is not None:
                  best_gallery_img = cv2.imread(os.path.join(dataset_dir, top1_match["filename"]))
                  hq, wq, _ = q_img.shape

                  query_corners = np.float32([[0, 0], [0, hq], [wq, hq], [wq, 0]]).reshape(-1, 1, 2)
                  projected_corners = cv2.perspectiveTransform(query_corners, H_est)

                  output_img = cv2.polylines(best_gallery_img.copy(), [np.int32(projected_corners)], True, (0, 255, 0), 4)

                  fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                  axes[0].imshow(cv2.cvtColor(q_img, cv2.COLOR_BGR2RGB))
                  axes[0].set_title(f"Query: {q_file}")
                  axes[0].axis("off")

                  axes[1].imshow(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB))
                  axes[1].set_title(f"Top 1 Fund: {top1_match['filename']}\nLandmark: {top1_match['landmark_id']}")
                  axes[1].axis("off")
                  plt.tight_layout()
                  plt.show()

    # --- FINAL STATISTICAL REPORT ---
    if total_processed_queries > 0:
        precision_at_1 = (top1_correct_count / total_processed_queries) * 100
        precision_at_5 = (top5_correct_count / total_processed_queries) * 100

        print("\n==================================================")
        print(f"Erfolgreich ausgewertete Queries: {total_processed_queries}")
        print(f"Top-1 Treffer (Precision@1):      {precision_at_1:.2f}%  ({top1_correct_count} von {total_processed_queries})")
        print(f"Top-5 Treffer (Precision@5):      {precision_at_5:.2f}%  ({top5_correct_count} von {total_processed_queries})")
        print("==================================================")
    else:
        print("Es konnten keine Queries erfolgreich ausgewertet werden.")

In [8]:
run_landmark_system(gallery_features, df_queries, "gldv2_micro/images")


Starte Suche für Query-Bild: e6f8a90e595ca5d3.jpg (Echte Landmark ID: 63593)

--- Top 5 Retrieval Ergebnisse ---
 Rang 1: Datei fd027f02bc1ce454.jpg | Landmark 174877 | Matches: 15 | ❌ ANDERE SEHENSWÜRDIGKEIT
 Rang 2: Datei fe51415602bbf932.jpg | Landmark 73406 | Matches: 13 | ❌ ANDERE SEHENSWÜRDIGKEIT
 Rang 3: Datei f9d4085988b767f9.jpg | Landmark 115903 | Matches: 13 | ❌ ANDERE SEHENSWÜRDIGKEIT
 Rang 4: Datei fd8bd9b9bb31e0e9.jpg | Landmark 8170 | Matches: 12 | ❌ ANDERE SEHENSWÜRDIGKEIT
 Rang 5: Datei f8beeb1284147b99.jpg | Landmark 40276 | Matches: 12 | ❌ ANDERE SEHENSWÜRDIGKEIT

Starte Suche für Query-Bild: ee2e41660c58f103.jpg (Echte Landmark ID: 59833)

--- Top 5 Retrieval Ergebnisse ---
 Rang 1: Datei ec045202835b6217.jpg | Landmark 59833 | Matches: 156 | ✅ KORREKT (Treffer)
 Rang 2: Datei e60bf8922d802c7f.jpg | Landmark 59833 | Matches: 30 | ✅ KORREKT (Treffer)
 Rang 3: Datei ee66d4366483535f.jpg | Landmark 59833 | Matches: 14 | ✅ KORREKT (Treffer)
 Rang 4: Datei f1cf0c6d4a061